# Stock Market Prediction using LSTM

Predict historical stock closing prices using an LSTM-based time-series model.

## 1. Import Libraries

The notebook uses NumPy, Pandas, Matplotlib, scikit-learn, TensorFlow/Keras, and yfinance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


## 2. Download Historical Data

This example uses Alphabet (Google) stock (`GOOGL`). The data is downloaded directly from Yahoo Finance through `yfinance`.

In [ ]:
ticker = 'GOOGL'
data = yf.download(ticker, start='2015-01-01', end='2025-01-01', auto_adjust=False, progress=False)

if data.empty:
    raise ValueError('No market data was downloaded. Check your internet connection or ticker symbol.')

data.head()


In [ ]:
close = data[['Close']].copy()
close = close.dropna()

print(close.shape)
close.head()


## 3. Visualize Closing Prices

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(close.index, close['Close'])
plt.title(f'{ticker} Historical Closing Price')
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.tight_layout()
plt.show()


## 4. Train-Test Split and Normalization

The final 20% of observations are kept as the test set. MinMaxScaler maps the training values to the range 0–1.

In [ ]:
values = close['Close'].values.reshape(-1, 1)
train_size = int(len(values) * 0.8)

train_values = values[:train_size]
test_values = values[train_size:]

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_values)
test_scaled = scaler.transform(test_values)


## 5. Create Sequential Time-Window Data

The model uses the previous 60 trading observations to predict the next closing price.

In [ ]:
window_size = 60

def create_sequences(series, window):
    X, y = [], []
    for i in range(window, len(series)):
        X.append(series[i-window:i, 0])
        y.append(series[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, window_size)

# Include the last training window before the test period so the first test prediction has context.
combined = np.concatenate([train_scaled[-window_size:], test_scaled])
X_test, y_test = create_sequences(combined, window_size)

X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)


## 6. Build the LSTM Model

In [ ]:
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    LSTM(50),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()


## 7. Train the Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()


## 8. Generate Test Predictions

In [ ]:
pred_scaled = model.predict(X_test, verbose=0)

predicted_prices = scaler.inverse_transform(pred_scaled)
actual_prices = scaler.inverse_transform(y_test.reshape(-1, 1))

rmse = np.sqrt(mean_squared_error(actual_prices, predicted_prices))
print(f'Test RMSE: ${rmse:.2f}')


## 9. Compare Actual and Predicted Prices

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(actual_prices, label='Actual Price')
plt.plot(predicted_prices, label='Predicted Price')
plt.title(f'{ticker} Actual vs Predicted Closing Prices')
plt.xlabel('Test Observation')
plt.ylabel('Closing Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()


## 10. Conclusion

The LSTM model learns temporal patterns from historical closing-price sequences and produces predictions for the held-out test period. RMSE is used to quantify prediction error, while the final plot provides a visual comparison between actual and predicted prices.

**Important:** The numerical RMSE printed by the notebook is the result that should be reported for the exact run, because the result depends on the downloaded data and training configuration.